In [2]:
# model_training.ipynb
# Train Logistic Regression, Decision Tree, and Random Forest models
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Clone repo
!git clone https://github.com/KeXen1/Student-Performance-ML-Project.git

%cd Student-Performance-ML-Project

In [ ]:
# Load cleaned dataset
data = pd.read_csv("data/processed/cleaned_data.csv")

display(data.head())
print(data.shape)

In [ ]:
# Create performance category target from G3
def grade_category(g3):
    if g3 < 10:
        return "Low"
    elif g3 < 15:
        return "Medium"
    else:
        return "High"

data["performance_category"] = data["G3"].apply(grade_category)

data["performance_category"].value_counts()
# Use early-semester data only
# Remove G3 because it is the final target
# Remove G2 because it is not early-semester data

X = data.drop(columns=["G3", "G2", "performance_category"], errors="ignore")
y = data["performance_category"]

print("Features used:")
print(X.columns.tolist())

In [ ]:
# Split data into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

In [ ]:
# Preprocessing
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

In [ ]:
# Train, evaluate, cross-validate, and save models
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": name,
        "Cross Validation Accuracy": cv_scores.mean(),
        "Test Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    model_filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(pipeline, f"models/{model_filename}")

    print("=" * 60)
    print(name)
    print("Cross-validation accuracy:", cv_scores.mean())
    print("Test accuracy:", accuracy)
    print()
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
# Save results
results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv("results/metrics.csv", index=False)

print("Training complete.")
print("Models saved in models/")
print("Metrics saved in results/metrics.csv")